# EEG Preprocessing simf10

This notebook processes iEEG data from the INTRA dataset. The recordings capture 200 epochs of epilesptic spikes.

## 🧪 - Experiment setup
- `ieeg`: 'raw' iEEG.

In [6]:
%matplotlib widget
import mindmap as mm

# reader
dm = mm.io.TutoManager("../../config.yml")
sr = dm.get_subject("intra-12")

# data
ieeg = sr.load("ieeg", task="spk")

## 🧬 - Data Processing
From Epochs to Average

In [7]:
ave = ieeg.average()

Normalize according to baseline

In [13]:
import numpy as np
import mne


def normalize_evoked_by_baseline(
    evoked: mne.Evoked,
    baseline: tuple[float, float],
 ) -> mne.Evoked:
    """
    Normalizes evoked data by the standard deviation of the baseline.

    Parameters
    ----------
    evoked: mne.Evoked
        Evoked object.

    baseline: tuple[float, float]
        (tmin, tmax) defining the basline period in seconds.
    """
    # create copy to avoid modifying original object in-place
    norm_evoked = evoked.copy()

    # select the baseline time indices
    idx_min, idx_max = evoked.time_as_index(baseline)

    # apply normalization
    norm_evoked.data = normalize_timeserie_by_baseline(evoked.data, (idx_min, idx_max))

    return norm_evoked


def normalize_timeserie_by_baseline(
    timeserie: np.ndarray,
    baseline: tuple[int, int],
    epsilon: float = 1e-15,
) -> np.ndarray:
    """
    Normalize timeserie by the standard deviation of the baseline.

    Parameters
    ----------
    timeserie: np.ndarray of shape (n_samples, n_timepoints)
        Timeserie.

    baseline: tuple[int, int]
        (idx_min, idx_max) defining the baseline period.

    epsilon: float
        Small number to check for zeros.

    Returns
    -------
    np.ndarray of shape (n_samples, n_timepoints)
        Baseline normalized timeseries.
    """
    timeserie = np.atleast_2d(timeserie)

    # extract baseline
    idx_min, idx_max = baseline
    baseline_data = timeserie[:, idx_min:idx_max]

    # calculate SD across baseline for each sample
    baseline_std = np.std(baseline_data, axis=1, keepdims=True)
    baseline_mean = np.mean(baseline_data, axis=1, keepdims=True)

    # avoid division by zero flat/bad samples
    baseline_std[baseline_std < epsilon] = 1.0

    # apply z-score
    return (timeserie - baseline_mean) / baseline_std

In [11]:
ave.time_as_index([0,2])

array([  0, 400])